# Ocean Wind Field Retrieval from Sentinel-1 SAR — Training and Validation

This notebook builds the training dataset, trains the ResNet wind-direction model and evaluates it.
Wind direction is retrieved from Sentinel-1 VV backscatter with a residual network
(Zanchetta & Zecchetto, 2021, *Remote Sensing of Environment* 253, 112178). Wind speed is obtained by
inverting the CMOD5.N geophysical model function with the retrieved direction.

| Step | Section | Output (Google Drive) |
|---|---|---|
| A | Build dataset from GeoTIFF patches and Copernicus Marine wind labels | `stageA/dataset_A.npz`, `qc_report.txt` |
| B | Scene-wise cross-validation | `cv_results/cv_summary.json`, `results_cv.png` |
| C | Final model training (3-model ensemble) | `final_model/` |
| D | Scene inference and validation (direction + CMOD5.N speed) | `scene_validation/validation_table.csv`, maps |

**Requirements**
- Exported patches in Google Drive: `<ROOT>/tifs/<scene_name>/*.tif` (from `gee/s1_patch_export.js`).
- A Copernicus Marine account (free) for the reference wind labels.
- GPU runtime for Sections 5–6 (*Runtime → Change runtime type → T4 GPU*).

Code, data and results are kept in Google Drive (`<ROOT>/sar_project` holds the files of `src/`). Run the sections in order;
after a restart only Section 1 has to be repeated. Clear all outputs before sharing the notebook
(*Edit → Clear all outputs*), as the sign-in output shows the account name.

## 1. Setup

In [ ]:
import os, sys, json, glob
import pandas as pd
from IPython.display import Image, display
from google.colab import drive
drive.mount("/content/drive")
!pip install -q copernicusmarine global-land-mask rasterio xarray netCDF4 xsarsea

ROOT      = "/content/drive/MyDrive/Resnet_wind_direction"   #@param {type:"string"}
CODE_DIR  = f"{ROOT}/sar_project"          # Python modules (src/)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
TIF_DIR   = f"{ROOT}/tifs"                 # one sub-folder per Sentinel-1 scene
DATA_DIR  = f"{ROOT}/stageA"               # dataset + Copernicus wind cache
CV_DIR    = f"{ROOT}/cv_results"           # cross-validation results
MODEL_DIR = f"{ROOT}/final_model"          # trained ensemble
VAL_DIR   = f"{ROOT}/scene_validation"     # per-scene wind fields, maps, validation table

### Training configuration
- `EXCLUDE`: scenes not used at all.
- `EVAL_ONLY`: scenes used only for testing: weak wind-streak texture (FFT anisotropy below ~0.18, see Section 4),
  streaks not aligned with the reference wind, or too few patches.
- Scenes without a Copernicus label (recent dates) and patches with wind below 3 m/s are left out automatically.
- `LOWPASS_PX`: Gaussian low-pass (σ in pixels of 100 m) that suppresses swell and speckle before the network.

In [ ]:
EXCLUDE    = ""   #@param {type:"string"}
EVAL_ONLY  = "durban,oman,san_diego,bahia,nothwest_medagascar,vancouver,vancouver_2,oaxaca,mistral,timor"   #@param {type:"string"}
LOWPASS_PX = 1.5                                             #@param {type:"number"}
FINAL_EXCLUDE = ",".join(s for s in [EXCLUDE, EVAL_ONLY] if s)   # final model: good-texture scenes only

## 2. Copernicus Marine sign-in
Required for Section 3 (wind labels) and Section 7 (reference wind). Enter your own credentials when prompted.

In [ ]:
import copernicusmarine
copernicusmarine.login()

## 3. Build the dataset
Reads every patch GeoTIFF, removes land / no-data patches, normalises each patch and attaches the Copernicus Marine
L4 hourly wind (0.125°), interpolated in time and space to the patch centre. Run again whenever scenes are added to `tifs/`.

In [ ]:
!python build_dataset.py --tif_root "{TIF_DIR}" --out_dir "{DATA_DIR}"

In [ ]:
print(open(f"{DATA_DIR}/qc_report.txt").read()[-1500:])

## 4. Data quality checks (optional)
**Patch montage** — visual check of selected scenes.

In [ ]:
SCENES_TO_VIEW = "Chennai,Gujarat"   #@param {type:"string"}
!python inspect_scenes.py --npz "{DATA_DIR}/dataset_A.npz" --out_dir "{CV_DIR}" --folders "{SCENES_TO_VIEW}"
for s in SCENES_TO_VIEW.split(","):
    p = f"{CV_DIR}/montage_{s.strip()}.png"
    if os.path.exists(p):
        display(Image(p, width=900))

**Texture analysis** — FFT anisotropy of the wind-streak band (0.8–2.5 km) per scene.
Scenes with mean anisotropy below about 0.2 carry little directional information and are candidates for `EVAL_ONLY`.

In [ ]:
!python texture_check.py --npz "{DATA_DIR}/dataset_A.npz" --out_dir "{CV_DIR}"
display(pd.read_csv(f"{CV_DIR}/texture_by_scene.csv"))

## 5. Scene-wise cross-validation (GPU)
Each fold holds out whole scenes, so every reported error is measured on scenes the model has not seen.
Training uses 8-fold dihedral augmentation, loss 1 − cos²(Δθ), Adam (lr 1e-3, weight decay 1e-4), dropout 0.3 and early stopping.

In [ ]:
!python train_resnet_v2.py --npz "{DATA_DIR}/dataset_A.npz" --out_dir "{CV_DIR}" --exclude_folders "{EXCLUDE}" --eval_only "{EVAL_ONLY}" --folds 5 --seeds 1 --lowpass_px {LOWPASS_PX}

In [ ]:
print(json.dumps(json.load(open(f"{CV_DIR}/cv_summary.json"))["summary"], indent=2))
display(pd.read_csv(f"{CV_DIR}/cv_metrics_by_scene.csv"))
display(Image(f"{CV_DIR}/results_cv.png", width=900))

## 6. Final model (GPU)
Trains an ensemble of three networks on all good-texture scenes. To publish a new model, copy the files of
`final_model/` (`model_*.pt`, `config.json`) to `models/final_model/` in the GitHub repository.

In [ ]:
!python train_final.py --npz "{DATA_DIR}/dataset_A.npz" --out_dir "{MODEL_DIR}" --exclude_folders "{FINAL_EXCLUDE}" --lowpass_px {LOWPASS_PX}

## 7. Scene inference and validation
For every scene: direction from the ensemble (with test-time augmentation), quality flag, 180° ambiguity removal
with the Copernicus direction, CMOD5.N wind speed and a wind-field map. Scenes used in training are marked *in-sample*.

In [ ]:
!python infer_scene.py --tif_dir "{TIF_DIR}" --model_dir "{MODEL_DIR}" --out_dir "{VAL_DIR}" --cache_dir "{DATA_DIR}/wind_cache"

In [ ]:
display(pd.read_csv(f"{VAL_DIR}/validation_table.csv"))
for p in sorted(glob.glob(f"{VAL_DIR}/map_*.png"))[:4]:
    display(Image(p, width=700))